In [ ]:


import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from torchmetrics import MeanSquaredError, R2Score
from torchmetrics.classification import MulticlassConfusionMatrix
import optuna


# Function for plotting predicted values against actual values for each lattice parameter (a, b, and c)
# Plots will show MSE and R2 values in upper left corner of each lattice parameter subplot by default
def PlotPredictions(Y_test_df, Y_pred_df, MSEVals, R2Vals, modelName, displayStatistics = True):
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # Creates 3 subplots for each lattice parameter
    for i, param in enumerate(Y_test_df.columns):
        ax = axes[i]

        # Plot each actual value against its predicted value as individual points
        ax.scatter(Y_test_df[param], Y_pred_df[f"{param}_pred"])

        # Create line of perfect fit where actual value = predicted value
        line = [Y_test_df[param].min(), Y_test_df[param].max()]
        ax.plot(line, line, 'r--')

        ax.set_xlabel("Actual Value")
        ax.set_ylabel("Predicted Value")
        ax.set_title(f"Lattice Parameter '{param}' (Å)")
        
        # Display MSE and R2 statistics for each lattice parameter if desired
        if displayStatistics:
            ax.text(
                0.05, 0.95,
                f"MSE={MSEVals[i]:.4f}\nR²={R2Vals[i]:.4f}",
                transform=ax.transAxes,
                verticalalignment='top',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor="gray")
            )
    
    fig.suptitle(f"Predicted vs. Actual Lattice Parameter Using {modelName}")
    
    plt.tight_layout()
    plt.show()

# Function for training neural network (NN) on current epoch
def train_epoch(model, train_dl, loss_fn, optimizer):
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(train_dl)

# Function for evaluating NN performance on current epoch
def eval_epoch(model, loader, loss_fn):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for xb, yb in loader:
            total_loss += loss_fn(model(xb), yb).item() * xb.size(0)
    return total_loss / len(loader.dataset)

# Create TabularDataset class to recast dataframe values as PyTorch tensors
class TabularDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.Y = torch.tensor(Y.values, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

# Initialize dataset paths
MPDatasetFeaturized = "<local_path>/FuelComp/MachineLearning/Dataset/MP_Dataset_Featurized.csv"
TrainingDatasetDir = "<local_path>/FuelComp/MachineLearning/Dataset/Training_Dataset.csv"

In [ ]:
# Read in featurized dataset and feature labels

df = pd.read_csv(MPDatasetFeaturized)
display(df.head())

featureLabels = joblib.load('FeatureLabels.joblib')
print(featureLabels)

In [ ]:
# Filter out materials with absurdly large lattice paramers and materials containing noble gases

latParamThreshold = 10

df = df[
    # (df['cubic'] == 1) &
    (df['a'] <= latParamThreshold) &
    (df['b'] <= latParamThreshold) &
    (df['c'] <= latParamThreshold)
]

display(df.head())

In [ ]:
# Create training/testing data for models

Y = df[['a', 'b', 'c']]

X = df[featureLabels]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 57)

Y_test_df = df_true = Y_test.reset_index(drop=True)

# Recast boolian values to integer as PyTorch NNs cannot handle boolian data
X_train = X_train.apply(lambda col: col.astype(int) if col.dtype == 'bool' else col)
X_test = X_test.apply(lambda col: col.astype(int) if col.dtype == 'bool' else col)

bool_cols = [col for col in X_train.columns if set(X_train[col].dropna().unique()) == {0,1} and "MagpieData" not in col]
cont_cols = [col for col in X_train.columns if col not in bool_cols]

print(bool_cols)

ct = ColumnTransformer([
    ('scale', StandardScaler(), cont_cols),
    ('passthrough', 'passthrough', bool_cols)
])

ct.fit(X_train)

X_train_scaled = pd.DataFrame(ct.transform(X_train))
X_test_scaled = pd.DataFrame(ct.transform(X_test))

# Create training and testing datasets
train_ds = TabularDataset(X_train_scaled, Y_train)
test_ds = TabularDataset(X_test_scaled, Y_test)

In [ ]:
# Create, train, and save NN model
# Technical name is multilayer perceptron, but that's too pretentious for code

# Load datasets for NN model training
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=64, shuffle=False)

# Define layers for NN
class TorchNN(nn.Module):
    def __init__(self, n_features, n_targets):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 600),
            nn.ReLU(),
            nn.Linear(600, 900),
            nn.ReLU(),
            nn.Dropout(0.0005),
            nn.Linear(900, 600),
            nn.ReLU(),
            nn.Dropout(0.0005),
            nn.Linear(600, 300),
            nn.ReLU(),
            nn.Linear(300, n_targets),          
        )
    def forward(self, x):
        return self.net(x)

# Initialize NN with n_features = len(featureLabels) (1 input neuron for every feature) and n_targets = 3 (1 output neuron for each lattice parameter)
model = TorchNN(n_features=len(featureLabels), n_targets=3)

# Initialize loss and optimizer metrics
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Set desired number of epochs, number of epochs to display loss information, and intialize random seed
n_epochs = 100
log_epochs = 10
torch.manual_seed(42)

# Train NN for desired number of epochs
for epoch in range(n_epochs):
    train_loss = train_epoch(model, train_dl, loss_fn, optimizer)
    val_loss = eval_epoch(model, test_dl, loss_fn)
    if epoch % log_epochs == 0:
        print(f"Epoch {epoch:02d} - train: {train_loss:.4f}, val: {val_loss:.4f}")

In [ ]:
# Create and display statistics for NN model

model.eval()

Y_pred_list = []
Y_true_list = []

with torch.no_grad():
    for xb, yb in test_dl:
        Y_pred_list.append(model(xb))
        Y_true_list.append(yb)

Y_pred = torch.cat(Y_pred_list, dim=0)
Y_true = torch.cat(Y_true_list, dim=0)

mse_metric = MeanSquaredError(num_outputs=3)
r2_metric = R2Score(multioutput='raw_values')

mse = mse_metric(Y_pred, Y_true)
r2 = r2_metric(Y_pred, Y_true)

Y_pred_df = pd.DataFrame(Y_pred.numpy(), columns=[f"{c}_pred" for c in Y_test.columns])

PlotPredictions(Y_test_df, Y_pred_df, mse, r2, "Multilayer Perceptron")

In [ ]:
# Bin test data and predictions and display confusion matrix
from torchmetrics.functional.classification import multiclass_confusion_matrix
import seaborn as sns

# def plot_confusion_matrix(cm, latParam):
#     plt.figure(figsize=(8, 6))
#     sns.heatmap(cm.numpy(), annot=True, fmt='d', cmap='Blues', cbar=False)
#     plt.title(f"Confusion Matrix for Lattice Parameter '{latParam}'")
#     plt.xlabel('Predicted')
#     plt.ylabel('True')
#     plt.show()

def plot_confusion_matrix(cm, latParam, bin_edges):
    # Define bin labels
    bin_labels = [f"{bin_edges[i]:.1f}–{bin_edges[i+1]:.1f}" for i in range(len(bin_edges) - 1)] + [f"{bin_edges[-2]:.1f}+"]

    # Create the heatmap
    plt.figure(figsize=(8, 6))
    ax = sns.heatmap(cm.numpy(), annot=True, fmt='d', cmap='Blues', cbar=False,
                     xticklabels=bin_labels, yticklabels=bin_labels)

    # Set labels and title
    ax.set_title(f"Confusion Matrix for Lattice Parameter '{latParam}'")
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

bin_edges = torch.tensor(torch.arange(2.5, 10.5, 0.5).tolist() + [float('inf')])

for i, latParam in enumerate(['a', 'b', 'c']):
    pred_bins = torch.bucketize(Y_pred[:, i], bin_edges, right=True)
    true_bins = torch.bucketize(Y_true[:, i], bin_edges, right=True)

    cm = multiclass_confusion_matrix(pred_bins, true_bins, len(bin_edges))

    plot_confusion_matrix(cm, latParam, bin_edges)

In [ ]:
# Perform hyperparameter tuning study to optimize NN model

def Objective(trial):
    n_layers = 4
    n_epochs = 80

    lr = trial.suggest_float('lr', 1e-6, 0.1, log=True)
    hidden_size1 = trial.suggest_int('hidden_size1', 300, 3000, step=300)
    hidden_size2 = trial.suggest_int('hidden_size2', 600, 6000, step=300)
    hidden_size3 = trial.suggest_int('hidden_size3', 300, 3000, step=300)
    hidden_size4 = trial.suggest_int('hidden_size4', 100, 1000, step=100)
    dropout = trial.suggest_float('dropout1', 0.0, 0.5)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-4, log=True)

    hidden_size = [hidden_size1, hidden_size2, hidden_size3, hidden_size4]

    layers = []
    in_features = len(featureLabels)
    for i in range(n_layers):
        layers.append(nn.Linear(in_features, hidden_size[i]))
        layers.append(nn.ReLU())
        if i < n_layers-1:
            layers.append(nn.Dropout(dropout))
        in_features = hidden_size[i]
    layers.append(nn.Linear(in_features, 3))
    model = nn.Sequential(*layers)

    train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
    val_dl = DataLoader(test_ds, batch_size=64, shuffle=False)

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_dl:
            optimizer.zero_grad()
            pred = model(xb)
            loss_fn(pred, yb).backward()
            optimizer.step()
        val_loss = eval_epoch(model, val_dl, loss_fn)
        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    
    return eval_epoch(val_dl)

study = optuna.create_study(direction='minimize')

study.optimize(Objective, n_trials=500, show_progress_bar=True)

print("Best hyperparameters:", study.best_trial.params)